In [5]:
import math
import torch

import ase
from ase.io import read

from quests.gpu.entropy import entropy

from fairchem.core import pretrained_mlip, FAIRChemCalculator

In [10]:
path = f"/home/grethel/dev/quests/examples/gap20/Graphite.xyz"
frames = read(path, index=":")
print(len(frames))

160


In [15]:
from fairchem.core import OCPCalculator
checkpoint_path = "/home/grethel/dev/fairchem_checkpoints/eqV2_153M_omat_mp_salex.pt"
calc = OCPCalculator(checkpoint_path=checkpoint_path)
print("-----------------model")
calc.trainer.model

INFO:root:amp: true
cmd:
  checkpoint_dir: /home/grethel/dev/quests/checkpoints/2026-01-26-22-09-04
  commit: ec91ac3
  identifier: ''
  logs_dir: /home/grethel/dev/quests/logs/wandb/2026-01-26-22-09-04
  print_every: 100
  results_dir: /home/grethel/dev/quests/results/2026-01-26-22-09-04
  seed: null
  timestamp_id: 2026-01-26-22-09-04
  version: 1.3.0
dataset:
  a2g_args:
    r_energy: true
    r_forces: true
    r_stress: true
  format: ase_db
  transforms:
    decompose_tensor:
      decomposition:
        stress_anisotropic:
          irrep_dim: 2
        stress_isotropic:
          irrep_dim: 0
      rank: 2
      tensor: stress
    element_references:
      file: /fsx-ocp-med/shared/alex-10M/alex-mp-norms-refs/element_references.pt
    normalizer:
      file: /fsx-ocp-med/shared/alex-10M/alex-mp-norms-refs/normalizers.pt
evaluation_metrics:
  metrics:
    energy:
    - mae
    - mae_density
    forces:
    - mae
    - forcesx_mae
    - forcesy_mae
    - forcesz_mae
    - cosine_

-----------------model


HydraModel(
  (backbone): EquiformerV2Backbone(
    (sphere_embedding): Embedding(96, 128)
    (distance_expansion): GaussianSmearing()
    (SO3_rotation): ModuleList(
      (0): SO3_Rotation(
        (mapping): CoefficientMappingModule(lmax_list=[6], mmax_list=[6])
      )
    )
    (mappingReduced): CoefficientMappingModule(lmax_list=[6], mmax_list=[3])
    (SO3_grid): (6, 6)
    (edge_degree_embedding): EdgeDegreeEmbedding(
      (SO3_rotation): ModuleList(
        (0): SO3_Rotation(
          (mapping): CoefficientMappingModule(lmax_list=[6], mmax_list=[6])
        )
      )
      (mappingReduced): CoefficientMappingModule(lmax_list=[6], mmax_list=[3])
      (source_embedding): Embedding(96, 128)
      (target_embedding): Embedding(96, 128)
      (rad_func): RadialFunction(
        (net): Sequential(
          (0): Linear(in_features=856, out_features=128, bias=True)
          (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (2): SiLU()
          (3): Linear(in_

In [28]:
# calc.trainer.model.backbone.norm.irreps_out
import torch
from e3nn.o3 import Irreps

model = calc.trainer.model  # or your actual loaded model

irreps_found = {}

for name, module in model.named_modules():
    # Try to introspect irreps attributes
    for attr in ["irreps_in", "irreps_out", "irreps"]:
        if hasattr(module, attr):
            try:
                irr = getattr(module, attr)
                # We check if it is an e3nn Irreps or convertible
                if isinstance(irr, Irreps) or (hasattr(irr, "__str__") and "irreps" in type(irr).__name__.lower()):
                    irreps_found[name + "." + attr] = irr
            except Exception as e:
                # Just skip if weird module
                pass

# Print the irreps we found
for k, v in irreps_found.items():
    print(f"{k}: {v}")

### Convert Equivariant Embeddings to Invariant

In [ ]:
import numpy as np
import torch

# --- Load your two npz files ---

orig = np.load("/home/grethel/dev/quests/embeddings/eqV2_31M_omat_mp_salex_Graphene_one_frame.npz", allow_pickle=True)
rot  = np.load("/home/grethel/dev/quests/embeddings/eqV2_31M_omat_mp_salex_Graphene_one_frame_rotated.npz", allow_pickle=True)

emb_orig_all = np.stack(orig['embeddings']).astype(np.float32)
emb_rot_all = np.stack(rot['embeddings']).astype(np.float32)
print(emb_orig_all.shape, emb_rot_all.shape)  # (404, 200, 3200)

# Flatten frames * atoms:
emb_orig = torch.from_numpy(emb_orig_all.reshape(-1, 3200))  # (404*200, 3200)
emb_rot  = torch.from_numpy(emb_rot_all.reshape(-1, 3200))   # same

print("Flattened shapes:", emb_orig.shape, emb_rot.shape)


# --- Reshape to (N_atoms, channels, irreps_components) ---

# channels = 128, irreps components = 25
emb_orig = emb_orig.view(-1, 128, 25)
emb_rot  = emb_rot.view(-1, 128, 25)


# --- Function to get invariant features ---
def equiformer_v2_to_invariant(emb: torch.Tensor) -> torch.Tensor:
    """
    Convert an Equiformer v2 equivariant embedding
    of shape (n_atoms, channels, 25) to invariant features
    shape (n_atoms, channels * 5).
    """
    # degree 0 scalar part
    inv_l0 = emb[:, :, 0]

    # norms of degrees l=1..4 blocks
    inv_l1 = torch.norm(emb[:, :, 1:4], dim=-1)
    inv_l2 = torch.norm(emb[:, :, 4:9], dim=-1)
    inv_l3 = torch.norm(emb[:, :, 9:16], dim=-1)
    inv_l4 = torch.norm(emb[:, :, 16:25], dim=-1)

    # concatenate per-channel invariants
    return torch.cat([inv_l0, inv_l1, inv_l2, inv_l3, inv_l4], dim=-1)

# --- Compute invariant embeddings ---

inv_orig = equiformer_v2_to_invariant(emb_orig)  # (200, 128*5)
inv_rot  = equiformer_v2_to_invariant(emb_rot)

print("Invariant shapes:", inv_orig.shape, inv_rot.shape)

# --- Compare invariants ---

diff_inv = torch.abs(inv_orig - inv_rot)
print("Max absolute diff (invariant):", diff_inv.max().item())
print("Mean absolute diff (invariant):", diff_inv.mean().item())

# expectation: differences only due to numerical rounding (~1e-6 or less)
if diff_inv.max() < 1e-5:
    print("✅ Invariant descriptors match within numeric tolerance.")
else:
    print("⚠️ Invariant descriptors have larger differences.")

# --- Raw equivariant difference ---

eqv_diff = torch.abs(emb_orig - emb_rot)
print("Max raw equivariant diff:", eqv_diff.max().item())

Raw shapes: (1, 200, 3200) (1, 200, 3200)
Flattened shapes: torch.Size([200, 3200]) torch.Size([200, 3200])
Invariant shapes: torch.Size([200, 81920]) torch.Size([200, 81920])
Max absolute diff (invariant): 64.49616241455078
Mean absolute diff (invariant): 0.013057248666882515
⚠️ Invariant descriptors have larger differences.
Max raw equivariant diff: 24.408016204833984


In [ ]:
import numpy as np
import torch

# --- Load your two npz files ---

orig = np.load("/home/grethel/dev/quests/embeddings/eqV2_86M_omat_mp_salex_Graphene_one_frame.npz", allow_pickle=True)
rot  = np.load("/home/grethel/dev/quests/embeddings/eqV2_86M_omat_mp_salex_Graphene_one_frame_rotated.npz", allow_pickle=True)

emb_orig_all = np.stack(orig['embeddings']).astype(np.float32)
emb_rot_all = np.stack(rot['embeddings']).astype(np.float32)
print(emb_orig_all.shape, emb_rot_all.shape)  

# Flatten frames * atoms:
emb_orig = torch.from_numpy(emb_orig_all.reshape(-1, 6272))  
emb_rot  = torch.from_numpy(emb_rot_all.reshape(-1, 6272))  
print("Flattened shapes:", emb_orig.shape, emb_rot.shape)


# --- Reshape to (N_atoms, channels, irreps_components) ---

# channels = 128, irreps components = 25
emb_orig = emb_orig.view(-1, 128, 49)
emb_rot  = emb_rot.view(-1, 128, 49)


# --- Function to get invariant features ---
def equiformer_v2_to_invariant(emb: torch.Tensor) -> torch.Tensor:
    """
    Convert an Equiformer v2 equivariant embedding of shape
    (n_atoms, channels, 49) to invariant features of shape
    (n_atoms, channels*7) = (n_atoms, 896).

    49 = 1 + 3 + 5 + 7 + 9 + 11 + 13 for l = 0..6
    """
    if emb.ndim != 3 or emb.shape[-1] != 49:
        raise ValueError(f"Expected (n_atoms, channels, 49), got {tuple(emb.shape)}")

    # l = 0 (scalar)
    inv_l0 = emb[:, :, 0]                     # (n_atoms, 128)

    # l = 1..6 (vector/tensor norms)
    inv_l1 = torch.norm(emb[:, :, 1:4],  dim=-1)
    inv_l2 = torch.norm(emb[:, :, 4:9],  dim=-1)
    inv_l3 = torch.norm(emb[:, :, 9:16], dim=-1)
    inv_l4 = torch.norm(emb[:, :, 16:25], dim=-1)
    inv_l5 = torch.norm(emb[:, :, 25:36], dim=-1)
    inv_l6 = torch.norm(emb[:, :, 36:49], dim=-1)

    # concatenate invariants
    inv = torch.cat(
        [inv_l0, inv_l1, inv_l2, inv_l3, inv_l4, inv_l5, inv_l6],
        dim=-1
    )

    return inv

# --- Compute invariant embeddings ---

inv_orig = equiformer_v2_to_invariant(emb_orig)  # (200, 128*5)
inv_rot  = equiformer_v2_to_invariant(emb_rot)

print("Invariant shapes:", inv_orig.shape, inv_rot.shape)

# --- Compare invariants ---

diff_inv = torch.abs(inv_orig - inv_rot)
print("Max absolute diff (invariant):", diff_inv.max().item())
print("Mean absolute diff (invariant):", diff_inv.mean().item())

# expectation: differences only due to numerical rounding (~1e-6 or less)
if diff_inv.max() < 1e-5:
    print("✅ Invariant descriptors match within numeric tolerance.")
else:
    print("⚠️ Invariant descriptors have larger differences.")

# --- Raw equivariant difference ---

eqv_diff = torch.abs(emb_orig - emb_rot)
print("Max raw equivariant diff:", eqv_diff.max().item())

Raw shapes: (1, 200, 6272) (1, 200, 6272)
Flattened shapes: torch.Size([200, 6272]) torch.Size([200, 6272])
Invariant shapes: torch.Size([200, 128]) torch.Size([200, 128])
Max absolute diff (invariant): 27.839027404785156
Mean absolute diff (invariant): 0.21004582941532135
❌ Invariants differ — something is wrong.
Max raw equivariant diff: 11.168083190917969
